In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import copy

# 1. Device Selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Reading and Preparation
df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
features = ['AT_solar_generation_actual', 'AT_wind_onshore_generation_actual', 'AT_load_actual_entsoe_transparency']
data = df[features].dropna()

# 3. Scaling
scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(data.values)

# 4. Sliding Window
def create_daily_sequences(data, lookback=48, horizon=24):
    X, y = [], []
    for i in range(len(data) - lookback - horizon + 1):
        X.append(data[i : (i + lookback), :])
        y.append(data[(i + lookback) : (i + lookback + horizon), :].flatten()) 
    return np.array(X), np.array(y)

X, y = create_daily_sequences(data_scaled, 48, 24)

# 5. GRU Model Class
class GRU_Model(nn.Module):
    def __init__(self):
        super(GRU_Model, self).__init__()
        self.gru = nn.GRU(input_size=3, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, 72)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

# ===========================================
# 6. TIME-SERIES CROSS VALIDATION
# ===========================================
n_splits = 3  # Veriyi 3 farklı parçaya bölerek test edeceğiz
tscv = TimeSeriesSplit(n_splits=n_splits)

fold = 1
wape_scores = []
mae_scores = []

print(f"--- ZAMAN SERİSİ ÇAPRAZ DOĞRULAMA ({n_splits} FOLD) BAŞLIYOR ---")

for train_index, test_index in tscv.split(X):
    print(f"\n[Fold {fold}] Eğitim Seti Boyutu: {len(train_index)}, Test Seti Boyutu: {len(test_index)}")

    # In each fold, we reserve the last 15% of the Train for Validation.
    fold_train_size = int(len(train_index) * 0.85)
    real_train_idx = train_index[:fold_train_size]
    val_idx = train_index[fold_train_size:]
    
    X_train, y_train = torch.from_numpy(X[real_train_idx]).float(), torch.from_numpy(y[real_train_idx]).float()
    X_val, y_val = torch.from_numpy(X[val_idx]).float(), torch.from_numpy(y[val_idx]).float()
    X_test, y_test = torch.from_numpy(X[test_index]).float(), torch.from_numpy(y[test_index]).float()

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=False)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)
    
    # WE COMPLETELY RESET the model and optimization for each fold so that it doesn't memorize it from the previous fold.
    model = GRU_Model().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_loss = float('inf')
    best_weights = copy.deepcopy(model.state_dict())
    patience = 5
    epochs_no_improve = 0
    
    # Eğitim Döngüsü

    for epoch in range(25):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(X_batch), y_batch).item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                break
                
    # En iyi ağırlıkları yükle ve test et

    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(device)).cpu().numpy()
        
    # Inverse scaling to calculate Consumption (Load) metrics only
    preds_mw = scaler.inverse_transform(preds.reshape(-1, 3)).reshape(preds.shape)
    y_test_mw = scaler.inverse_transform(y_test.numpy().reshape(-1, 3)).reshape(y_test.shape)

    preds_load = preds_mw[:, 2::3] 
    y_test_load = y_test_mw[:, 2::3]

    mae = mean_absolute_error(y_test_load, preds_load)
    wape = (mae / np.mean(y_test_load)) * 100
    
    print(f"Fold {fold} Tamamlandı -> MAE: {mae:.2f} MW | WAPE: %{wape:.2f}")
    
    mae_scores.append(mae)
    wape_scores.append(wape)
    fold += 1

print("\n=============================================")
print(f"FINAL CROSS-VALIDATION RESULT")
print(f"Ortalama MAE : {np.mean(mae_scores):.2f} MW")
print(f"Ortalama WAPE: %{np.mean(wape_scores):.2f}")
print("==============================================")